# Query cost lab

Four architectures, one workload, priced side by side.

This is the module that turns "BigQuery is expensive" into something you can defend. Every
number here comes from the course's own price list, so it is reproducible without an account.


## 0. Setup


In [ ]:
import sys
from pathlib import Path

# The course's pure core modules. No installation and no cloud SDK — that is
# the whole point of the split.
COURSE_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd().parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))

from cost_model import (
    BIGQUERY_ONDEMAND_PER_TB,
    BIGQUERY_SLOT_HOUR,
    compare_architectures,
    estimate,
    format_comparison,
    monthly_runway,
)
from pipeline_logic import (
    aggregate_events,
    bucket_by_window,
    dedup_key,
    deduplicate,
    estimated_scan_bytes,
    partition_for_date,
    partition_for_event,
)

print("core modules on the path:", (COURSE_ROOT / "project" / "core").exists())


## 1. The four architectures

`compare_architectures()` prices the same workload four ways and returns them **cheapest
first**.

Read the output carefully: two rows are `$0.0000`, because the default `streaming_hours=0.0`
means no streaming was configured. **Never read a `$0.00` streaming row as free.**


In [ ]:
print(format_comparison(compare_architectures()))


## 2. Give the streaming rows a real number

Now the comparison means something. Note that Dataproc is cheaper per vCPU-hour than Dataflow —
and that does not make Dataproc the right answer.


In [ ]:
options = compare_architectures(streaming_hours=24, batch_runs=30)
print(format_comparison(options))
print()
for name, cost in options:
    print(f"{name:<38} ${cost:>10.4f}")


## 3. The 100× partitioning multiplier

The single biggest lever in BigQuery cost, and it costs nothing to apply.

Partitioning is a **range** operation on a date column. A query with a date predicate skips
whole partitions at the metadata level — they are never opened.


In [ ]:
print(f"{'prune ratio':>12} {'TB scanned':>12} {'cost':>12}")
print("-" * 38)
for ratio in (1.0, 0.1, 0.01, 0.001):
    tb = 10.0 * ratio
    cost = estimate("bigquery", scanned_tb=tb, apply_free_tier=False).total
    print(f"{ratio:>12} {tb:>12.2f} {cost:>12.4f}")


## 4. On-demand versus flat-rate

Two different units, and the crossover is a real number you can compute.

- **On-demand**: $6.25 per TB scanned. You pay for what you read.
- **Flat-rate**: $0.06 per slot-hour. You pay for capacity, whether or not it is busy.

The crossover: at what scan volume does 1000 slot-hours cost the same as the on-demand scan?


In [ ]:
flat = estimate("bigquery", slot_hours=1000, apply_free_tier=False).total
print(f"flat-rate, 1000 slot-hours: ${flat:.2f}")
print()
print(f"on-demand equivalent:       ${flat:.2f}")
print(f"= {flat / BIGQUERY_ONDEMAND_PER_TB:.1f} TB scanned")
print()
print("Below that volume, on-demand wins.")
print("Above it, flat-rate wins — provided the slots are actually busy.")
print("Reserve for the average and you queue at the peaks.")


## 5. The three billing units

The one that surprises people is storage, because it is metered separately from everything
else — and it is what makes a materialized view a trade rather than a free win.


In [ ]:
print("=== SCAN: on-demand bytes ===")
print(f"10 TB scan: ${estimate('bigquery', scanned_tb=10.0, apply_free_tier=False).total:.2f}")
print()
print("=== SLOTS: compute capacity ===")
print(f"1000 slot-hours: ${estimate('bigquery', slot_hours=1000, apply_free_tier=False).total:.2f}")
print()
print("=== STORAGE: data at rest ===")
for gb in (100, 1024, 10240):
    cost = estimate("bigquery", storage_gb=gb, apply_free_tier=False).total
    print(f"{gb:>6,} GB-month: ${cost:>10.2f}")
print()
print(f"Dataflow monthly runway: ${monthly_runway('dataflow'):.2f}")
print(f"Compute Engine monthly runway: ${monthly_runway('compute-engine'):.2f}")


## 6. A dashboard, priced both ways

The materialized-view argument, in numbers. The read frequency is what decides it.


In [ ]:
SCAN_TB, READS, DAYS = 2.0, 200, 30

logical = estimate("bigquery", scanned_tb=SCAN_TB * READS * DAYS,
                   apply_free_tier=False).total
materialized = estimate("bigquery", scanned_tb=SCAN_TB * DAYS, storage_gb=50,
                        apply_free_tier=False).total

print(f"logical view:      ${logical:>12,.2f}")
print(f"materialized view: ${materialized:>12,.2f}")
print(f"difference:        ${logical - materialized:>12,.2f}")
print()
print(f"break-even at about {materialized / (SCAN_TB * BIGQUERY_ONDEMAND_PER_TB):.0f} reads per month.")


## 7. Guardrails

`maximumBytesBilled` turns a budget incident into a failed job. The cost of the failure is a
retry; the cost of the success is unbounded.


In [ ]:
MAX_BYTES = 10_000_000_000
MAX_TB = MAX_BYTES / 1024**4

print(f"maximumBytesBilled: {MAX_BYTES:,} bytes")
print(f"that is {MAX_TB:.4f} TB")
print(f"priced at $6.25/TB: ${MAX_TB * BIGQUERY_ONDEMAND_PER_TB:.2f}")
print()
print("Above that, the job FAILS. Which is the correct behaviour at 3am.")


## What you should be able to say now

1. Why two rows in the architecture comparison are `$0.0000`.
2. The 100× partitioning win, in dollars.
3. Where the on-demand and flat-rate lines cross.
4. Why storage is a separate billing unit and what it makes expensive.
5. What `maximumBytesBilled` does and why it belongs on every job.
